In [1]:
# ==============================================================================
# ConMamba-ASR — Q1 NEW EXPERIMENTS SUITE (Fast Dual-T4 Parallel Runner)
# ------------------------------------------------------------------------------
# Removes old redundant runs and executes the exact Q1 reviewer-requested suite:
#   1. Complete Table IV (unchunked, fixed, chunked, overlap_energy) + Peak VRAM
#   2. Fine-Tuned Foundation Baseline: Whisper-Small + LoRA (5 epochs, FP16)
#   3. Noise Robustness Stress Test: Babble+Ambient SNR sweep (Clean,20,10,5,0dB)
#   4. Architectural Ablations (Parallel T4x2): Unidirectional, K=7, No-CTC (a=0)
#   5. Exact Encoder Latency/VRAM Scaling (5s..60s) + Paired Bootstrap & p-value
#   6. Auto-generated IEEE TASLP LaTeX (booktabs) tables
# ==============================================================================

import os, sys, re, glob, json, time, math, shutil, hashlib, platform, subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

# ==============================================================================
# 0. EXPERIMENT CONFIGURATION (EDIT ONLY THIS BLOCK)
# ==============================================================================
# Point this to your saved checkpoints dataset (containing conmamba_small_s7775
# and/or conformer_small_s7775). If found, skips retraining & runs inference only!
EXISTING_CKPTS_DIR = None       # e.g., "/kaggle/input/ejust-trained-ckpts"
RETRAIN_BASELINES_IF_MISSING = False  # Set True ONLY if checkpoints aren't saved

# --- Toggle New Q1 Experiments ---
RUN_WHISPER_LORA       = True   # ~25-35 min on 1x T4 (Huge Q1 value)
RUN_ARCH_ABLATIONS     = True   # Runs in parallel on Dual-T4
RUN_LONG_ABLATION_FULL = True   # Completes Table IV + Unchunked (Inference only)
RUN_NOISE_STRESS_TEST  = True   # SNR sweep Clean/20/10/5/0 dB (Inference only)
RUN_LATENCY_BENCHMARK  = True   # 5s -> 60s Encoder Latency & Peak VRAM

# --- Speed & Training Settings ---
ABLATION_EPOCHS        = 80     # Fast (~1.8h/model on T4). Use 110 for full parity
VALID_SEARCH_INTERVAL  = 40     # Fast: beam-search validation only twice
TEST_BEAM_SIZE         = 5      # Matches paper (2x faster than beam=10)
TIME_LIMIT_H           = 11.2   # Safe stop before Kaggle's 12h limit

# --- Architectural Ablations to Run (Proves Contribution #3) ---
# Each entry: (run_name, base_template, seed, overrides_dict)
ARCH_ABLATIONS = [
    ("conmamba_unidir",  "conmamba_small", 7775, {"bidirectional": False}),
    ("conmamba_k7",      "conmamba_small", 7775, {"kernel_size": 7}),
    ("conmamba_no_ctc",  "conmamba_small", 7775, {"ctc_weight": 0.0}),
    # Optional (uncomment if running in a second session):
    # ("bimamba_noconv", "conmamba_small", 7775, {"encoder_module": "mamba"}),
    # ("conmamba_n8",    "conmamba_small", 7775, {"d_state": 8}),
]

# --- Whisper-Small LoRA Config ---
WHISPER_MODEL  = "openai/whisper-small"
WHISPER_EPOCHS = 5
WHISPER_LR     = 1e-3
WHISPER_BS     = 16             # Effective batch size = 32 with grad_accum=2

# --- Long-Form & Noise Eval Config ---
LONG_STRATEGIES = ["unchunked", "fixed", "chunked", "overlap_energy"]
SNR_LEVELS_DB   = [None, 20, 10, 5, 0]  # None = Clean

# --- Known Paper Numbers (Fallback if baseline checkpoints aren't attached) ---
KNOWN_BASELINES = {
    "conmamba_small": {
        "model": "ConMamba-Small (Ours)", "params_m": 12.1,
        "test_wer": 13.85, "test_cer": 4.44,
        "test_long_wer": 14.65, "test_long_cer": 4.57, "rtf_long": 0.0438,
    },
    "conformer_small": {
        "model": "Conformer-Small", "params_m": 12.5,
        "test_wer": 14.49, "test_cer": 4.77,
        "test_long_wer": 15.38, "test_long_cer": 4.82, "rtf_long": 0.0624,
    },
    "whisper_zero_shot": {
        "model": "Whisper-Small (Zero-Shot)", "params_m": 241.7,
        "test_wer": 71.63, "test_cer": 33.64,
        "test_long_wer": 53.62, "test_long_cer": 21.16, "rtf_long": 0.0966,
    },
}

WORK = "/kaggle/working"
REPO = f"{WORK}/Mamba-ASR"
RESULTS_ROOT = f"{WORK}/q1_new_results"
os.makedirs(RESULTS_ROOT, exist_ok=True)

# ==============================================================================
# 1. ENVIRONMENT & DEPENDENCY SETUP
# ==============================================================================
import numpy as np
import pandas as pd
import torch

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 1, "No GPU detected! Enable Accelerator -> GPU T4 x2."
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | GPUs: {NUM_GPUS}x {torch.cuda.get_device_name(0)}")

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install([
    "speechbrain==1.0.3", "sentencepiece", "einops", "soundfile",
    "packaging", "ninja", "jiwer", "scipy",
])
pip_install(["causal-conv1d", "mamba-ssm", "--no-build-isolation"])
if RUN_WHISPER_LORA:
    pip_install(["transformers", "accelerate", "peft"])

if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-q", "https://github.com/xi-j/Mamba-ASR.git", REPO])

# ==============================================================================
# 2. WRITE ejust_tools.py (COMPATIBILITY SHIM + SAFE ABLATION YAML PATCHER)
# ==============================================================================
ejust_tools_code = r'''
from pathlib import Path
import re

SHIM = """# --- compatibility shim (EJUST) ---
try:
    from causal_conv1d.cpp_functions import (
        causal_conv1d_fwd_function as _cc_new_fwd,
        causal_conv1d_bwd_function as _cc_new_bwd,
    )
    def _cc_fwd(x, weight, bias, seq_idx, silu):
        return _cc_new_fwd(x, weight, bias, seq_idx, None, None, silu)
    def _cc_bwd(x, weight, bias, dout, seq_idx, dx, silu):
        dx, dweight, dbias, _ = _cc_new_bwd(x, weight, bias, dout, seq_idx, None, None, dx, False, silu)
        return dx, dweight, dbias
except ImportError:
    import causal_conv1d_cuda
    def _cc_fwd(x, weight, bias, seq_idx, silu):
        try:
            return causal_conv1d_cuda.causal_conv1d_fwd(x, weight, bias, seq_idx, None, None, silu)
        except TypeError:
            return causal_conv1d_cuda.causal_conv1d_fwd(x, weight, bias, seq_idx, silu)
    def _cc_bwd(x, weight, bias, dout, seq_idx, dx, silu):
        try:
            r = causal_conv1d_cuda.causal_conv1d_bwd(x, weight, bias, dout, seq_idx, None, None, dx, False, silu)
            return r[0], r[1], r[2]
        except TypeError:
            return causal_conv1d_cuda.causal_conv1d_bwd(x, weight, bias, dout, seq_idx, dx, silu)
"""

LN_OLD = "    from mamba_ssm.ops.triton.layernorm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
LN_NEW = (
    "    try:\n"
    "        from mamba_ssm.ops.triton.layernorm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
    "    except ImportError:\n"
    "        from mamba_ssm.ops.triton.layer_norm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
)

def patch_repo(repo):
    repo = Path(repo)
    p = repo / "modules/mamba/selective_scan_interface.py"
    s = p.read_text()
    if "_cc_fwd" not in s:
        s = re.sub(r"causal_conv1d_cuda\.causal_conv1d_fwd\((x, conv1d_weight)", r"_cc_fwd(\1", s)
        s = s.replace("causal_conv1d_cuda.causal_conv1d_bwd(\n", "_cc_bwd(\n")
        s = s.replace("import causal_conv1d_cuda\n", SHIM, 1)
        p.write_text(s)
    for f in ["modules/mamba/bimamba.py", "modules/mamba/mamba_blocks.py"]:
        p = repo / f
        s = p.read_text()
        if "except ImportError:" not in s and LN_OLD in s:
            p.write_text(s.replace(LN_OLD, LN_NEW, 1))
    p_tasr = repo / "modules/TransformerASR.py"
    if p_tasr.exists():
        s_tasr = p_tasr.read_text()
        old_tgt = "if self.num_decoder_layers > 0:\n            tgt = self.custom_tgt_module(tgt)"
        new_tgt = "if self.num_decoder_layers > 0 and tgt is not None:\n            tgt = self.custom_tgt_module(tgt)"
        if old_tgt in s_tasr:
            p_tasr.write_text(s_tasr.replace(old_tgt, new_tgt, 1))
    return True

def make_hparams(template, out_path, *, data_folder, csv_dir, bpe_model, output_folder,
                 epochs, warmup, valid_search_interval, test_beam_size=5,
                 batch_size=32, lr=0.0008, num_workers=2, grad_accumulation_factor=1,
                 long_strategies="chunked", snr_levels="", skip_train=False,
                 ablation_overrides=None):
    s = Path(template).read_text()
    extra_header = (
        f"data_folder: {data_folder}\n"
        f"bpe_model: {bpe_model}\n"
        f"long_strategies: {long_strategies}\n"
        f"snr_levels: \"{snr_levels}\""
    )
    rep = [
        (r"^data_folder: !PLACEHOLDER.*$", extra_header),
        (r"^skip_train: .*$", f"skip_train: {skip_train}"),
        (r"^output_folder: .*$", f"output_folder: {output_folder}"),
        (r"^train_csv: .*$", f"train_csv: {csv_dir}/train.csv"),
        (r"^valid_csv: .*$", f"valid_csv: {csv_dir}/dev.csv"),
        (r"^test_csv:\n(    - .*\n)+",
         f"test_csv:\n    - {csv_dir}/test.csv\n    - {csv_dir}/test_long.csv\n"),
        (r"^number_of_epochs: .*$", f"number_of_epochs: {epochs}"),
        (r"^grad_accumulation_factor: .*$", f"grad_accumulation_factor: {grad_accumulation_factor}"),
        (r"^num_workers: .*$", f"num_workers: {num_workers}"),
        (r"^precision: .*$", "precision: fp16"),
        (r"^lr_adam: .*$", f"lr_adam: {lr}"),
        (r"^batch_size: .*$", f"batch_size: {batch_size}"),
        (r"^dynamic_batching: .*$", "dynamic_batching: False"),
        (r"^valid_dataloader_opts:\n    batch_size: 1$", "valid_dataloader_opts:\n    batch_size: 32"),
        (r"^test_dataloader_opts:\n    batch_size: 1$", "test_dataloader_opts:\n    batch_size: 16"),
        (r"^output_neurons: .*$", "output_neurons: 500"),
        (r"^valid_search_interval: .*$", f"valid_search_interval: {valid_search_interval}"),
        (r"^test_beam_size: .*$", f"test_beam_size: {test_beam_size}"),
        (r"^    n_warmup_steps: .*$", f"    n_warmup_steps: {warmup}"),
    ]
    for pat, new in rep:
        s, n = re.subn(pat, new, s, count=1, flags=re.M)
        assert n == 1, f"{Path(template).name}: pattern not found: {pat}"

    # Apply architectural ablation overrides safely (supports top-level & nested keys)
    if ablation_overrides:
        for k, v in ablation_overrides.items():
            val_str = str(v)
            pat = rf"^(\s*{re.escape(k)}:\s*).*$"
            s_new, n = re.subn(pat, rf"\g<1>{val_str}", s, flags=re.M)
            if n == 0:
                print(f"[WARN] Ablation key '{k}' not found in {Path(template).name}; skipping override.")
            else:
                print(f"[ABLATION] {Path(out_path).name}: set {k} -> {val_str} ({n} match)")
                s = s_new

    Path(out_path).write_text(s)
    return out_path
'''
Path(f"{WORK}/ejust_tools.py").write_text(ejust_tools_code)
sys.path.insert(0, WORK); sys.path.insert(0, REPO)
import ejust_tools
ejust_tools.patch_repo(REPO)
print("Patched Mamba-ASR successfully.")

# ==============================================================================
# 3. WRITE Q1 TRAINER + NOISE STRESS TEST + FULL LONG-FORM EVALUATOR
# ==============================================================================
train_script = r'''#!/usr/bin/env python3
import os, sys, json, time, hashlib, logging
from pathlib import Path
import numpy as np
import torch
import soundfile as sf
import torchaudio

if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]

torch.backends.cudnn.benchmark = True
import speechbrain as sb
from hyperpyyaml import load_hyperpyyaml
from speechbrain.utils.distributed import if_main_process

logger = logging.getLogger(__name__)

# --- Realistic Deterministic Noise Generator (Babble + Pink Ambient) ---
def _pink_noise(n_samples, gen):
    white = torch.randn(n_samples, generator=gen)
    fft = torch.fft.rfft(white)
    freqs = torch.fft.rfftfreq(n_samples)
    freqs[0] = 1.0
    pink = torch.fft.irfft(fft / torch.sqrt(freqs), n=n_samples)
    return pink / (torch.std(pink) + 1e-8)

def mix_realistic_noise(wav, uid, snr_db, babble_pool=None):
    """Deterministic babble + pink ambient noise at exact target SNR (dB)."""
    if snr_db is None:
        return wav
    seed = int(hashlib.md5(f"{uid}_{snr_db}".encode()).hexdigest()[:8], 16)
    gen = torch.Generator().manual_seed(seed)
    n = wav.size(-1)

    pink = _pink_noise(n, gen)
    if babble_pool is not None and len(babble_pool) >= 3:
        idx = torch.randint(0, len(babble_pool), (3,), generator=gen).tolist()
        babble = torch.zeros(n)
        for k in idx:
            src = babble_pool[k]
            if src.numel() == 0:
                continue
            reps = (n // src.numel()) + 2
            tiled = src.repeat(reps)
            offset = int(torch.randint(0, max(1, tiled.numel() - n), (1,), generator=gen).item())
            babble = babble + tiled[offset:offset + n]
        babble = babble / (torch.std(babble) + 1e-8)
        noise = 0.65 * babble + 0.35 * pink
    else:
        noise = pink

    noise = noise.to(wav.device, dtype=wav.dtype)
    sig_pow = torch.mean(wav ** 2) + 1e-12
    noise_pow = torch.mean(noise ** 2) + 1e-12
    target_noise_pow = sig_pow / (10.0 ** (float(snr_db) / 10.0))
    noisy = wav + torch.sqrt(target_noise_pow / noise_pow) * noise
    peak = torch.max(torch.abs(noisy))
    if peak > 0.99:
        noisy = noisy * (0.99 / peak)
    return noisy

# --- Long-Form Segmentation Strategies (Complete Table IV) ---
def segment_fixed(wav, sr=16000, max_dur=4.0):
    max_s = int(max_dur * sr)
    return [wav[i:i + max_s] for i in range(0, wav.size(-1), max_s)]

def segment_energy(wav, sr=16000, max_dur=4.5, min_dur=2.5, overlap_dur=0.0, hop_size=160, frame_size=400):
    total = wav.size(-1)
    max_s, min_s, ov_s = int(max_dur * sr), int(min_dur * sr), int(overlap_dur * sr)
    if total <= max_s or total < frame_size:
        return [wav]
    unfolded = wav.unfold(-1, frame_size, hop_size)
    energy = torch.sqrt(torch.mean(unfolded ** 2, dim=-1) + 1e-12)
    chunks, start = [], 0
    while start < total:
        if total - start <= max_s:
            chunks.append(wav[start:])
            break
        f_start = (start + min_s) // hop_size
        f_end = min((start + max_s) // hop_size, energy.size(-1))
        if f_end <= f_start:
            split = min(start + max_s, total)
        else:
            split = (f_start + torch.argmin(energy[f_start:f_end]).item()) * hop_size
        if split <= start:
            split = min(start + max_s, total)
        chunks.append(wav[start:split])
        next_start = max(split - ov_s, start + min_s // 2)
        start = next_start if ov_s > 0 else split
    return chunks

def merge_overlapping_texts(texts, max_ov_words=5):
    if not texts:
        return ""
    merged = texts[0].split()
    for txt in texts[1:]:
        words = txt.split()
        if not words:
            continue
        best_k = 0
        limit = min(max_ov_words, len(merged), len(words))
        for k in range(limit, 0, -1):
            if merged[-k:] == words[:k]:
                best_k = k
                break
        merged.extend(words[best_k:])
    return " ".join(merged)

class ASR(sb.core.Brain):
    def _decode_chunks_batched(self, chunks, current_epoch, searcher, batch_size=16):
        valid_chunks = [c for c in chunks if c.size(0) >= 400]
        if not valid_chunks:
            return []
        texts = []
        for i in range(0, len(valid_chunks), batch_size):
            sub = valid_chunks[i:i + batch_size]
            max_l = max(c.size(0) for c in sub)
            padded = torch.zeros(len(sub), max_l, device=self.device)
            lens = torch.zeros(len(sub), device=self.device)
            for j, c in enumerate(sub):
                padded[j, :c.size(0)] = c
                lens[j] = c.size(0) / max_l
            feats = self.hparams.compute_features(padded)
            feats = self.modules.normalize(feats, lens, epoch=current_epoch)
            src = self.modules.CNN(feats)
            if hasattr(self.modules.Transformer, "encode"):
                enc = self.modules.Transformer.encode(src, wav_len=lens)
            else:
                enc, _ = self.modules.Transformer(src, None, lens)
            hyps, _, _, _ = searcher(enc.detach(), lens)
            for h in hyps:
                t = self.tokenizer.decode_ids(h).strip()
                if t:
                    texts.append(t)
        return texts

    def decode_long_utterance(self, wav, current_epoch, searcher, strategy):
        sr = self.hparams.sample_rate
        if strategy == "fixed":
            chunks = segment_fixed(wav, sr=sr, max_dur=4.0)
            texts = self._decode_chunks_batched(chunks, current_epoch, searcher)
            full_text = " ".join(texts).strip()
        elif strategy == "overlap_energy":
            chunks = segment_energy(wav, sr=sr, max_dur=4.5, min_dur=2.5, overlap_dur=0.5)
            texts = self._decode_chunks_batched(chunks, current_epoch, searcher)
            full_text = merge_overlapping_texts(texts)
        else:  # chunked (energy/silence)
            chunks = segment_energy(wav, sr=sr, max_dur=4.5, min_dur=2.5, overlap_dur=0.0)
            texts = self._decode_chunks_batched(chunks, current_epoch, searcher)
            full_text = " ".join(texts).strip()
        return self.tokenizer.encode_as_ids(full_text)

    def compute_forward(self, batch, stage):
        batch = batch.to(self.device)
        wavs, wav_lens = batch.sig
        tokens_bos, _ = batch.tokens_bos

        if stage == sb.Stage.TEST:
            self._test_audio_sec += sum(
                wav_lens[k].item() * wavs.size(1) / self.hparams.sample_rate
                for k in range(wavs.size(0))
            )
            snr_db = getattr(self.hparams, "active_snr_db", None)
            if snr_db is not None:
                babble_pool = getattr(self, "_babble_pool", None)
                wavs = wavs.clone()
                for b in range(wavs.size(0)):
                    b_samp = int(round(wav_lens[b].item() * wavs.size(1)))
                    wavs[b, :b_samp] = mix_realistic_noise(
                        wavs[b, :b_samp], batch.id[b], snr_db, babble_pool
                    )

        strategy = getattr(self.hparams, "active_long_strategy", "standard")
        current_epoch = self.hparams.epoch_counter.current
        sr = self.hparams.sample_rate

        # Handle chunked long-form without running full-length encoder unnecessarily
        if stage == sb.Stage.TEST and strategy in ("fixed", "chunked", "overlap_energy"):
            searcher = self.hparams.test_search
            hyps = []
            for b in range(wavs.size(0)):
                b_samp = int(round(wav_lens[b].item() * wavs.size(1)))
                b_wav = wavs[b, :b_samp]
                hyps.append(self.decode_long_utterance(b_wav, current_epoch, searcher, strategy))
            # Return dummy logits so compute_objectives computes WER/CER cleanly
            return None, None, wav_lens, hyps

        # Standard or Unchunked Full-Sequence forward pass (with OOM safety)
        try:
            feats = self.hparams.compute_features(wavs)
            feats = self.modules.normalize(feats, wav_lens, epoch=current_epoch)
            if stage == sb.Stage.TRAIN and hasattr(self.hparams, "fea_augment"):
                feats, fea_lens = self.hparams.fea_augment(feats, wav_lens)
                tokens_bos = self.hparams.fea_augment.replicate_labels(tokens_bos)
            src = self.modules.CNN(feats)
            enc_out, pred = self.modules.Transformer(src, tokens_bos, wav_lens, pad_idx=self.hparams.pad_index)
            p_ctc = self.hparams.log_softmax(self.modules.ctc_lin(enc_out))
            p_seq = self.hparams.log_softmax(self.modules.seq_lin(pred))

            hyps = None
            is_valid_search = stage == sb.Stage.VALID and (
                current_epoch % self.hparams.valid_search_interval == 0
                or current_epoch == self.hparams.number_of_epochs
            )
            if is_valid_search:
                hyps, _, _, _ = self.hparams.valid_search(enc_out.detach(), wav_lens)
            elif stage == sb.Stage.TEST:
                hyps, _, _, _ = self.hparams.test_search(enc_out.detach(), wav_lens)
            return p_ctc, p_seq, wav_lens, hyps
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            self._oom_count = getattr(self, "_oom_count", 0) + 1
            hyps = [[] for _ in range(wavs.size(0))]
            return None, None, wav_lens, hyps

    def compute_objectives(self, predictions, batch, stage):
        p_ctc, p_seq, wav_lens, hyps = predictions
        ids = batch.id
        tokens_eos, tokens_eos_lens = batch.tokens_eos
        tokens, tokens_lens = batch.tokens

        if p_ctc is None or p_seq is None:
            # Chunked long-form or OOM fallback during TEST
            if hyps is not None:
                predicted_words = [self.tokenizer.decode_ids(h).split(" ") if h else [] for h in hyps]
                target_words = [w.split(" ") for w in batch.wrd]
                self.wer_metric.append(ids, predicted_words, target_words)
                self.cer_metric.append(ids, predicted_words, target_words)
            return torch.tensor(0.0, device=self.device)

        if stage == sb.Stage.TRAIN and hasattr(self.hparams, "fea_augment"):
            rep = self.hparams.fea_augment.replicate_labels
            tokens, tokens_lens = rep(tokens), rep(tokens_lens)
            tokens_eos, tokens_eos_lens = rep(tokens_eos), rep(tokens_eos_lens)
        loss_seq = self.hparams.seq_cost(p_seq, tokens_eos, length=tokens_eos_lens).sum()
        loss_ctc = self.hparams.ctc_cost(p_ctc, tokens, wav_lens, tokens_lens).sum()
        loss = self.hparams.ctc_weight * loss_ctc + (1 - self.hparams.ctc_weight) * loss_seq

        if stage != sb.Stage.TRAIN:
            current_epoch = self.hparams.epoch_counter.current
            do_wer = (
                stage == sb.Stage.TEST
                or current_epoch % self.hparams.valid_search_interval == 0
                or current_epoch == self.hparams.number_of_epochs
            )
            if do_wer and hyps is not None:
                predicted_words = [self.tokenizer.decode_ids(h).split(" ") for h in hyps]
                target_words = [w.split(" ") for w in batch.wrd]
                self.wer_metric.append(ids, predicted_words, target_words)
                self.cer_metric.append(ids, predicted_words, target_words)
            self.acc_metric.append(p_seq, tokens_eos, tokens_eos_lens)
        return loss

    def on_evaluate_start(self, max_key=None, min_key=None):
        super().on_evaluate_start()
        ckpts = self.checkpointer.find_checkpoints(max_key=max_key, min_key=min_key)
        ckpt = sb.utils.checkpoints.average_checkpoints(ckpts, recoverable_name="model")
        self.hparams.model.load_state_dict(ckpt, strict=True)
        self.hparams.model.eval()

    def on_stage_start(self, stage, epoch):
        self._t0 = time.time()
        self._test_audio_sec = 0.0
        self._oom_count = 0
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        if stage != sb.Stage.TRAIN:
            self.acc_metric = self.hparams.acc_computer()
            self.wer_metric = self.hparams.error_rate_computer()
            self.cer_metric = self.hparams.error_rate_computer(split_tokens=True)

    def _log_efficiency(self, stage, epoch):
        if not if_main_process():
            return
        elapsed = round(time.time() - self._t0, 2)
        rtf = round(elapsed / self._test_audio_sec, 4) if (stage == sb.Stage.TEST and self._test_audio_sec > 0) else None
        rec = {
            "stage": str(stage).split(".")[-1],
            "epoch": epoch,
            "seconds": elapsed,
            "audio_sec": round(self._test_audio_sec, 1) if stage == sb.Stage.TEST else None,
            "rtf": rtf,
            "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 2**20) if torch.cuda.is_available() else None,
            "oom_count": getattr(self, "_oom_count", 0),
            "test_name": getattr(self.hparams, "test_name", None) if stage == sb.Stage.TEST else None,
        }
        with open(os.path.join(self.hparams.output_folder, "efficiency.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")

    def on_stage_end(self, stage, stage_loss, epoch):
        self._log_efficiency(stage, epoch)
        stage_stats = {"loss": stage_loss}
        if stage == sb.Stage.TRAIN:
            self.train_stats = stage_stats
            return
        if self.acc_metric.correct + self.acc_metric.total > 0:
            stage_stats["ACC"] = self.acc_metric.summarize()
        current_epoch = self.hparams.epoch_counter.current
        do_wer = (
            stage == sb.Stage.TEST
            or current_epoch % self.hparams.valid_search_interval == 0
            or current_epoch == self.hparams.number_of_epochs
        )
        if do_wer:
            stage_stats["WER"] = self.wer_metric.summarize("error_rate")
            stage_stats["CER"] = self.cer_metric.summarize("error_rate")
        if stage == sb.Stage.VALID:
            lr = self.hparams.noam_annealing.current_lr
            self.hparams.train_logger.log_stats(
                stats_meta={"epoch": epoch, "lr": lr, "steps": self.optimizer_step},
                train_stats=self.train_stats, valid_stats=stage_stats,
            )
            self.checkpointer.save_and_keep_only(
                meta={"ACC": stage_stats.get("ACC", 0.0), "epoch": epoch},
                max_keys=["ACC"], num_to_keep=self.hparams.avg_checkpoints,
            )
        elif stage == sb.Stage.TEST:
            self.hparams.train_logger.log_stats(
                stats_meta={"Epoch": self.hparams.epoch_counter.current, "test": self.hparams.test_name},
                test_stats=stage_stats,
            )
            if if_main_process():
                with open(self.hparams.test_wer_file, "w", encoding="utf-8") as w:
                    self.wer_metric.write_stats(w)
                with open(self.hparams.test_wer_file.replace("wer_", "cer_"), "w", encoding="utf-8") as w:
                    self.cer_metric.write_stats(w)

    def on_fit_batch_end(self, batch, outputs, loss, should_step):
        if should_step:
            self.hparams.noam_annealing(self.optimizer)

def dataio_prepare(hparams, tokenizer):
    data_folder = hparams["data_folder"]
    load = lambda p: sb.dataio.dataset.DynamicItemDataset.from_csv(csv_path=p, replacements={"data_root": data_folder})
    train_data = load(hparams["train_csv"])
    valid_data = load(hparams["valid_csv"]).filtered_sorted(sort_key="duration")
    test_datasets = {Path(p).stem: load(p).filtered_sorted(sort_key="duration") for p in hparams["test_csv"]}
    datasets = [train_data, valid_data] + list(test_datasets.values())
    valtest = [valid_data] + list(test_datasets.values())

    def read(wav):
        x, sr = sf.read(wav, dtype="float32", always_2d=False)
        return torch.from_numpy(x)

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline(wav):
        return read(wav)
    sb.dataio.dataset.add_dynamic_item(valtest, audio_pipeline)

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline_train(wav):
        sig = read(wav)
        if "speed_perturb" in hparams:
            sig = hparams["speed_perturb"](sig.unsqueeze(0)).squeeze(0)
        return sig
    sb.dataio.dataset.add_dynamic_item([train_data], audio_pipeline_train)

    @sb.utils.data_pipeline.takes("wrd")
    @sb.utils.data_pipeline.provides("wrd", "tokens_list", "tokens_bos", "tokens_eos", "tokens")
    def text_pipeline(wrd):
        yield wrd
        tokens_list = tokenizer.encode_as_ids(wrd)
        yield tokens_list
        yield torch.LongTensor([hparams["bos_index"]] + tokens_list)
        yield torch.LongTensor(tokens_list + [hparams["eos_index"]])
        yield torch.LongTensor(tokens_list)
    sb.dataio.dataset.add_dynamic_item(datasets, text_pipeline)
    sb.dataio.dataset.set_output_keys(datasets, ["id", "sig", "wrd", "tokens_bos", "tokens_eos", "tokens"])
    return train_data, valid_data, test_datasets

def build_babble_pool(train_csv, data_folder, max_clips=40):
    import pandas as pd
    df = pd.read_csv(train_csv)
    df = df[df["duration"] >= 2.0].head(max_clips)
    pool = []
    for p in df["wav"].tolist():
        wav_path = p.replace("$data_root", data_folder)
        if os.path.exists(wav_path):
            x, _ = sf.read(wav_path, dtype="float32", always_2d=False)
            pool.append(torch.from_numpy(x))
    return pool

if __name__ == "__main__":
    hparams_file, run_opts, overrides = sb.parse_arguments(sys.argv[1:])
    with open(hparams_file, encoding="utf-8") as fin:
        hparams = load_hyperpyyaml(fin, overrides)
    sb.utils.distributed.ddp_init_group(run_opts)
    sb.create_experiment_directory(
        experiment_directory=hparams["output_folder"],
        hyperparams_to_save=hparams_file, overrides=overrides,
    )
    tokenizer = hparams["tokenizer"]
    tokenizer.load(hparams["bpe_model"])
    train_data, valid_data, test_datasets = dataio_prepare(hparams, tokenizer)
    hparams["test_search"] = hparams["valid_search"]
    hparams["test_search"].beam_size = hparams["test_beam_size"]

    asr_brain = ASR(
        modules=hparams["modules"], opt_class=hparams["Adam"],
        hparams=hparams, run_opts=run_opts, checkpointer=hparams["checkpointer"],
    )
    asr_brain.tokenizer = tokenizer

    n_params = sum(p.numel() for p in asr_brain.hparams.model.parameters())
    Path(hparams["output_folder"], "params.json").write_text(json.dumps({"params": n_params}))

    if not hparams["skip_train"]:
        asr_brain.fit(
            asr_brain.hparams.epoch_counter, train_data, valid_data,
            train_loader_kwargs=hparams["train_dataloader_opts"],
            valid_loader_kwargs=hparams["valid_dataloader_opts"],
        )

    os.makedirs(hparams["output_wer_folder"], exist_ok=True)
    strategies = [s.strip() for s in str(hparams.get("long_strategies", "chunked")).split(",") if s.strip()]
    snr_raw = str(hparams.get("snr_levels", "")).strip()
    snr_list = [int(x) for x in snr_raw.split(",") if x.strip()] if snr_raw else []

    # 1. Standard Short Test & Long-Form Decoding Ablations
    for name, data in test_datasets.items():
        asr_brain.hparams.active_snr_db = None
        if "long" in name:
            for strat in strategies:
                asr_brain.hparams.active_long_strategy = strat
                asr_brain.hparams.test_name = f"{name}_{strat}"
                asr_brain.hparams.test_wer_file = os.path.join(hparams["output_wer_folder"], f"wer_{name}_{strat}.txt")
                test_opts = dict(hparams["test_dataloader_opts"])
                test_opts["batch_size"] = 1
                asr_brain.evaluate(data, max_key="ACC", test_loader_kwargs=test_opts)
        else:
            asr_brain.hparams.active_long_strategy = "standard"
            asr_brain.hparams.test_name = name
            asr_brain.hparams.test_wer_file = os.path.join(hparams["output_wer_folder"], f"wer_{name}.txt")
            asr_brain.evaluate(data, max_key="ACC", test_loader_kwargs=dict(hparams["test_dataloader_opts"]))

    # 2. Noise Robustness Stress Test (Babble + Pink Ambient SNR Sweep)
    if snr_list and "test" in test_datasets:
        asr_brain._babble_pool = build_babble_pool(hparams["train_csv"], hparams["data_folder"])
        asr_brain.hparams.active_long_strategy = "standard"
        for snr in snr_list:
            asr_brain.hparams.active_snr_db = snr
            asr_brain.hparams.test_name = f"test_snr{snr}"
            asr_brain.hparams.test_wer_file = os.path.join(hparams["output_wer_folder"], f"wer_test_snr{snr}.txt")
            asr_brain.evaluate(test_datasets["test"], max_key="ACC", test_loader_kwargs=dict(hparams["test_dataloader_opts"]))
'''
Path(f"{REPO}/train_egy_q1.py").write_text(train_script)

# ==============================================================================
# 4. LOCATE DATASET (ejust-prepared-v1)
# ==============================================================================
bpe_candidates = sorted(glob.glob("/kaggle/input/**/tokenizer/bpe500.model", recursive=True))
assert bpe_candidates, "ejust-prepared-v1 not found! Add Input -> Datasets -> ejust-prepared-v1."
BPE = bpe_candidates[0]
DATA = os.path.dirname(os.path.dirname(BPE))
print(f"Dataset root: {DATA}")

GLOBAL_T0 = time.time()
TIME_LIMIT_SEC = int(TIME_LIMIT_H * 3600)

# ==============================================================================
# 5. HELPER: RUN SPEECHBRAIN JOB (TRAINING OR INFERENCE-ONLY)
# ==============================================================================
def run_sb_job(run_id, template_name, seed, gpu_id, *,
               epochs, long_strategies, snr_levels, skip_train=False,
               ablation_overrides=None):
    out_dir = f"{RESULTS_ROOT}/{run_id}"

    # Check if pre-trained checkpoint exists in EXISTING_CKPTS_DIR
    if EXISTING_CKPTS_DIR and not os.path.isdir(out_dir):
        hits = sorted(glob.glob(f"{EXISTING_CKPTS_DIR}/**/{run_id}", recursive=True))
        if hits:
            shutil.copytree(hits[0], out_dir)
            print(f"[GPU {gpu_id}] Restored checkpoint for {run_id} from {hits[0]}")

    has_ckpt = bool(glob.glob(f"{out_dir}/save/CKPT*"))
    if skip_train and not has_ckpt:
        print(f"[GPU {gpu_id}] SKIP {run_id}: skip_train=True but no checkpoint found in {out_dir}/save.")
        return run_id, -1

    hp_path = f"{WORK}/{run_id}.yaml"
    ejust_tools.make_hparams(
        f"{REPO}/hparams/S2S/{template_name}.yaml", hp_path,
        data_folder=DATA, csv_dir=DATA, bpe_model=BPE, output_folder=out_dir,
        epochs=epochs, warmup=min(20000, int(epochs * 819 * 0.3)),
        valid_search_interval=VALID_SEARCH_INTERVAL, test_beam_size=TEST_BEAM_SIZE,
        batch_size=32, grad_accumulation_factor=1, num_workers=2,
        long_strategies=",".join(long_strategies),
        snr_levels=",".join(str(x) for x in snr_levels if x is not None),
        skip_train=skip_train, ablation_overrides=ablation_overrides,
    )

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cmd = [
        sys.executable, "train_egy_q1.py", hp_path,
        "--precision", "fp16", "--seed", str(seed), "--ckpt_interval_minutes", "30",
    ]
    print(f"[GPU {gpu_id}] START {run_id} (skip_train={skip_train}) -> {out_dir}")
    t0 = time.time()
    log_path = f"{WORK}/{run_id}_stdout.txt"
    with open(log_path, "w") as logf:
        proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            logf.write(line)
            if any(k in line for k in ("train_logger", "Epoch", "ERROR", "Traceback", "ABLATION")):
                print(f"[{run_id}|GPU{gpu_id}] {line}", end="")
            if time.time() - GLOBAL_T0 > TIME_LIMIT_SEC:
                print(f"[GPU {gpu_id}] TIME LIMIT — stopping {run_id} cleanly.")
                proc.terminate()
                break
        proc.wait()
    print(f"[GPU {gpu_id}] DONE {run_id} | exit={proc.returncode} | {(time.time()-t0)/60:.1f} min")
    return run_id, proc.returncode

# ==============================================================================
# 6. WHISPER-SMALL LoRA FINE-TUNING WORKER (RUNS ON DEDICATED GPU)
# ==============================================================================
def run_whisper_lora_experiment(gpu_id=0):
    out_json = f"{RESULTS_ROOT}/whisper_lora_results.json"
    if os.path.exists(out_json):
        print(f"[GPU {gpu_id}] SKIP Whisper-LoRA (already finished).")
        return json.loads(Path(out_json).read_text())

    whisper_script = r'''
import os, re, json, time, sys
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from torch.utils.data import Dataset
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from jiwer import wer as jiwer_wer, cer as jiwer_cer
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, pipeline,
)
from peft import LoraConfig, get_peft_model

DATA, MODEL_ID, OUT_DIR, EPOCHS, LR, BS = sys.argv[1], sys.argv[2], sys.argv[3], int(sys.argv[4]), float(sys.argv[5]), int(sys.argv[6])
os.makedirs(OUT_DIR, exist_ok=True)

def norm_ar(s):
    s = re.sub(r"[\u0617-\u061A\u064B-\u0652\u0640]", "", str(s))
    s = re.sub(r"[^\w\s\u0600-\u06FF]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Arabic", task="transcribe")

class EjustWhisperDS(Dataset):
    def __init__(self, csv_path, data_root):
        df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
        self.wavs = [p.replace("$data_root", data_root) for p in df["wav"].tolist()]
        self.texts = df["wrd"].tolist()
    def __len__(self):
        return len(self.wavs)
    def __getitem__(self, idx):
        x, _ = sf.read(self.wavs[idx], dtype="float32", always_2d=False)
        feats = processor.feature_extractor(x, sampling_rate=16000).input_features[0]
        labels = processor.tokenizer(self.texts[idx]).input_ids
        return {"input_features": feats, "labels": labels}

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.use_cache = False
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.enable_input_require_grads()

lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
model = get_peft_model(model, lora_cfg)
total_p = sum(p.numel() for p in model.parameters()) / 1e6
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"[Whisper-LoRA] Total params: {total_p:.1f}M | Trainable LoRA: {train_p:.2f}M")

train_ds = EjustWhisperDS(f"{DATA}/train.csv", DATA)
args = Seq2SeqTrainingArguments(
    output_dir=f"{OUT_DIR}/ckpts",
    per_device_train_batch_size=BS,
    gradient_accumulation_steps=max(1, 32 // BS),
    learning_rate=LR,
    warmup_ratio=0.1,
    num_train_epochs=EPOCHS,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    logging_steps=200,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

t_train = time.time()
trainer = Seq2SeqTrainer(
    args=args, model=model, train_dataset=train_ds,
    data_collator=DataCollatorSpeechSeq2Seq(processor=processor),
)
trainer.train()
train_min = round((time.time() - t_train) / 60.0, 1)
print(f"[Whisper-LoRA] Fine-tuning complete in {train_min} min.")

# Merge LoRA into base weights for fair zero-overhead inference RTF
merged_model = model.merge_and_unload().eval().half().cuda()
pipe = pipeline(
    "automatic-speech-recognition",
    model=merged_model, tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0, torch_dtype=torch.float16, chunk_length_s=30,
)

res = {
    "model": "Whisper-Small + LoRA (FT)", "params_m": round(total_p, 1),
    "trainable_m": round(train_p, 2), "train_min": train_min,
}
for split in ["test", "test_long"]:
    df = pd.read_csv(f"{DATA}/{split}.csv", dtype=str, keep_default_na=False)
    wavs = [p.replace("$data_root", DATA) for p in df["wav"].tolist()]
    refs = [norm_ar(r) for r in df["wrd"].tolist()]
    total_audio = df["duration"].astype(float).sum()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    outs = pipe(wavs, batch_size=16, generate_kwargs={"language": "arabic", "task": "transcribe"})
    elapsed = time.time() - t0
    hyps = [norm_ar(o["text"]) for o in outs]
    res[f"{split}_wer"] = round(jiwer_wer(refs, hyps) * 100, 2)
    res[f"{split}_cer"] = round(jiwer_cer(refs, hyps) * 100, 2)
    res["rtf_test" if split == "test" else "rtf_long"] = round(elapsed / total_audio, 4)
    res[f"peak_mb_{split}"] = round(torch.cuda.max_memory_allocated() / 2**20)
    print(f"[Whisper-LoRA] {split:10s} | WER={res[f'{split}_wer']:.2f}% | CER={res[f'{split}_cer']:.2f}% | RTF={res['rtf_test' if split=='test' else 'rtf_long']:.4f}")

Path(f"{OUT_DIR}/whisper_lora_results.json").write_text(json.dumps(res, indent=2))
'''
    w_path = f"{WORK}/run_whisper_lora.py"
    Path(w_path).write_text(whisper_script)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cmd = [sys.executable, w_path, DATA, WHISPER_MODEL, RESULTS_ROOT, str(WHISPER_EPOCHS), str(WHISPER_LR), str(WHISPER_BS)]
    print(f"[GPU {gpu_id}] START Whisper-Small LoRA Fine-Tuning ({WHISPER_EPOCHS} epochs)...")
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if any(k in line for k in ("[Whisper-LoRA]", "loss", "ERROR", "Traceback")):
            print(line, end="")
    proc.wait()
    return json.loads(Path(out_json).read_text()) if os.path.exists(out_json) else None

# ==============================================================================
# 7. SCHEDULE ALL JOBS IN PARALLEL ACROSS DUAL-T4 GPUS
# ==============================================================================
jobs = []

# Job Group 1: Full Table IV + Noise Stress Test on Existing Baselines (Fast Inference)
if RUN_LONG_ABLATION_FULL or RUN_NOISE_STRESS_TEST:
    l_strats = LONG_STRATEGIES if RUN_LONG_ABLATION_FULL else ["chunked"]
    s_lvls = [x for x in SNR_LEVELS_DB if x is not None] if RUN_NOISE_STRESS_TEST else []
    for base_m in ["conmamba_small", "conformer_small"]:
        jobs.append((
            "sb", f"{base_m}_s7775", base_m, 7775,
            dict(epochs=110, long_strategies=l_strats, snr_levels=s_lvls,
                 skip_train=not RETRAIN_BASELINES_IF_MISSING, ablation_overrides=None)
        ))

# Job Group 2: Whisper-Small LoRA Fine-Tuning (Runs concurrently with Ablations!)
if RUN_WHISPER_LORA:
    jobs.append(("whisper", "whisper_lora", None, None, {}))

# Job Group 3: Architectural Ablations (Proves Contribution #3)
if RUN_ARCH_ABLATIONS:
    for run_name, tmpl, seed, ov in ARCH_ABLATIONS:
        jobs.append((
            "sb", f"{run_name}_s{seed}", tmpl, seed,
            dict(epochs=ABLATION_EPOCHS, long_strategies=["chunked"], snr_levels=[],
                 skip_train=False, ablation_overrides=ov)
        ))

def _dispatch(job, gpu_id):
    kind, rid, tmpl, seed, kw = job
    if kind == "whisper":
        return rid, run_whisper_lora_experiment(gpu_id=gpu_id)
    return run_sb_job(rid, tmpl, seed, gpu_id, **kw)

print(f"\n=== Dispatching {len(jobs)} Q1 Jobs across {NUM_GPUS} GPU(s) ===")
with ProcessPoolExecutor(max_workers=NUM_GPUS) as ex:
    futs = {ex.submit(_dispatch, j, idx % NUM_GPUS): j[1] for idx, j in enumerate(jobs)}
    for fut in as_completed(futs):
        name = futs[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"[ERROR] Job {name} failed: {e}")

# ==============================================================================
# 8. ENCODER LATENCY & PEAK VRAM SCALING BENCHMARK (5s -> 60s ON T4)
# ==============================================================================
latency_rows = []
if RUN_LATENCY_BENCHMARK:
    print("\n=== Running Encoder Latency & Peak VRAM Scaling Benchmark (5s - 60s) ===")
    from hyperpyyaml import load_hyperpyyaml
    for m_name in ["conmamba_small", "conformer_small"]:
        hp_tmp = f"{WORK}/bench_{m_name}.yaml"
        ejust_tools.make_hparams(
            f"{REPO}/hparams/S2S/{m_name}.yaml", hp_tmp,
            data_folder=DATA, csv_dir=DATA, bpe_model=BPE, output_folder=f"{WORK}/tmp_bench",
            epochs=1, warmup=100, valid_search_interval=1, skip_train=True,
        )
        with open(hp_tmp) as f:
            hp = load_hyperpyyaml(f)
        feat_fn = hp["compute_features"]
        cnn = hp["modules"]["CNN"].cuda().eval().half()
        trans = hp["modules"]["Transformer"].cuda().eval().half()

        for dur in [5, 15, 30, 45, 60]:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            wav = torch.randn(1, int(dur * 16000), device="cuda")
            lens = torch.tensor([1.0], device="cuda")
            try:
                with torch.no_grad(), torch.cuda.amp.autocast():
                    feats = feat_fn(wav).half()
                    # Warmup
                    for _ in range(3):
                        src = cnn(feats)
                        _ = trans.encode(src, wav_len=lens) if hasattr(trans, "encode") else trans(src, None, lens)
                    torch.cuda.synchronize()
                    times = []
                    for _ in range(8):
                        t0 = time.perf_counter()
                        src = cnn(feats)
                        _ = trans.encode(src, wav_len=lens) if hasattr(trans, "encode") else trans(src, None, lens)
                        torch.cuda.synchronize()
                        times.append((time.perf_counter() - t0) * 1000.0)
                latency_rows.append({
                    "model": m_name, "duration_s": dur,
                    "latency_ms": round(float(np.median(times)), 1),
                    "peak_vram_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1),
                })
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                latency_rows.append({"model": m_name, "duration_s": dur, "latency_ms": "OOM", "peak_vram_mb": "OOM"})
        del cnn, trans; torch.cuda.empty_cache()

# ==============================================================================
# 9. PARSING, STATISTICAL SIGNIFICANCE (BOOTSTRAP + MAPSSWE), & LATEX TABLES
# ==============================================================================
def parse_wer_summary(path):
    if not os.path.exists(path):
        return None
    m = re.search(r"%WER\s+([\d\.]+)", open(path, encoding="utf-8").readline())
    return float(m.group(1)) if m else None

def parse_per_utt_errors(path):
    """Parse SpeechBrain wer_*.txt per-utterance (errors, ref_words) for Bootstrap/MAPSSWE."""
    if not os.path.exists(path):
        return {}
    out = {}
    pat = re.compile(r"^(\S+),\s*%WER\s*[\d\.]+\s*\[\s*(\d+)\s*/\s*(\d+)")
    for line in open(path, encoding="utf-8"):
        m = pat.match(line.strip())
        if m:
            out[m.group(1)] = (int(m.group(2)), int(m.group(3)))
    return out

def load_eff_map(out_dir):
    eff_file = Path(out_dir, "efficiency.jsonl")
    m = {}
    if eff_file.exists():
        for line in eff_file.read_text().splitlines():
            r = json.loads(line)
            if r.get("stage") == "TEST" and r.get("test_name"):
                m[r["test_name"]] = r
    return m

# --- A. Main + Fine-Tuned Whisper Comparison Table ---
w_lora_path = f"{RESULTS_ROOT}/whisper_lora_results.json"
w_lora = json.loads(Path(w_lora_path).read_text()) if os.path.exists(w_lora_path) else None

main_rows = [
    KNOWN_BASELINES["whisper_zero_shot"],
    *([w_lora] if w_lora else []),
    KNOWN_BASELINES["conformer_small"],
    KNOWN_BASELINES["conmamba_small"],
]
df_main = pd.DataFrame(main_rows)
print("\n" + "=" * 90)
print("NEW TABLE A: CONMAMBA vs ZERO-SHOT & FINE-TUNED FOUNDATION MODEL (WHISPER-LoRA)")
print("=" * 90)
print(df_main[["model", "params_m", "test_wer", "test_cer", "test_long_wer", "test_long_cer", "rtf_long"]].to_string(index=False))

# --- B. Architectural Ablation Table (Contribution #3) ---
ablation_rows = [{
    "Variant": "ConMamba-Small (Full: Bi-SSM + K=31 + CTC a=0.3)",
    "Params (M)": 12.1, "Test WER (%)": 13.85, "Test CER (%)": 4.44, "Long WER (%)": 14.65,
}]
for run_name, _, seed, _ in ARCH_ABLATIONS:
    od = f"{RESULTS_ROOT}/{run_name}_s{seed}"
    wd = f"{od}/wer_output"
    if os.path.isdir(wd):
        pm = round(json.loads(Path(od, "params.json").read_text())["params"] / 1e6, 1) if Path(od, "params.json").exists() else "-"
        ablation_rows.append({
            "Variant": run_name, "Params (M)": pm,
            "Test WER (%)": parse_wer_summary(f"{wd}/wer_test.txt"),
            "Test CER (%)": parse_wer_summary(f"{wd}/cer_test.txt"),
            "Long WER (%)": parse_wer_summary(f"{wd}/wer_test_long_chunked.txt"),
        })
df_abl = pd.DataFrame(ablation_rows)
print("\n" + "=" * 90)
print("NEW TABLE B: ARCHITECTURAL ABLATION STUDY (PROVES CONTRIBUTION #3)")
print("=" * 90)
print(df_abl.to_string(index=False))

# --- C. Complete Long-Form Decoding Ablation (Fixes Table IV + Unchunked) ---
long_rows = []
for base_m in ["conmamba_small", "conformer_small"]:
    od = f"{RESULTS_ROOT}/{base_m}_s7775"
    wd = f"{od}/wer_output"
    eff = load_eff_map(od)
    for strat in LONG_STRATEGIES:
        w = parse_wer_summary(f"{wd}/wer_test_long_{strat}.txt")
        if w is not None:
            e = eff.get(f"test_long_{strat}", {})
            long_rows.append({
                "Model": base_m, "Strategy": strat,
                "Long WER (%)": w,
                "Long CER (%)": parse_wer_summary(f"{wd}/cer_test_long_{strat}.txt"),
                "RTF": e.get("rtf"), "Peak VRAM (MB)": e.get("peak_gpu_mb"), "OOMs": e.get("oom_count", 0),
            })
if long_rows:
    df_long = pd.DataFrame(long_rows)
    print("\n" + "=" * 90)
    print("NEW TABLE C: COMPLETE TABLE IV (LONG-FORM DECODING + UNCHUNKED)")
    print("=" * 90)
    print(df_long.to_string(index=False))

# --- D. Noise Robustness Stress Test Table (SNR Sweep) ---
noise_rows = []
for base_m in ["conmamba_small", "conformer_small"]:
    wd = f"{RESULTS_ROOT}/{base_m}_s7775/wer_output"
    if not os.path.isdir(wd):
        continue
    row = {"Model": base_m, "Clean": parse_wer_summary(f"{wd}/wer_test.txt")}
    for snr in [20, 10, 5, 0]:
        row[f"{snr} dB"] = parse_wer_summary(f"{wd}/wer_test_snr{snr}.txt")
    noise_rows.append(row)
if noise_rows:
    df_noise = pd.DataFrame(noise_rows)
    print("\n" + "=" * 90)
    print("NEW TABLE D: NOISE ROBUSTNESS STRESS TEST (BABBLE + PINK AMBIENT WER %)")
    print("=" * 90)
    print(df_noise.to_string(index=False))

# --- E. Latency & VRAM Scaling Table (5s -> 60s) ---
if latency_rows:
    df_lat = pd.DataFrame(latency_rows)
    print("\n" + "=" * 90)
    print("NEW TABLE E: ENCODER LATENCY (ms) & PEAK VRAM (MB) SCALING (T4 GPU)")
    print("=" * 90)
    print(df_lat.pivot(index="duration_s", columns="model", values=["latency_ms", "peak_vram_mb"]).to_string())

# --- F. Statistical Significance (Paired Bootstrap 95% CI + Wilcoxon p-value) ---
cm_utt = parse_per_utt_errors(f"{RESULTS_ROOT}/conmamba_small_s7775/wer_output/wer_test.txt")
cf_utt = parse_per_utt_errors(f"{RESULTS_ROOT}/conformer_small_s7775/wer_output/wer_test.txt")
common_ids = sorted(set(cm_utt.keys()) & set(cf_utt.keys()))
if len(common_ids) > 100:
    from scipy.stats import wilcoxon
    e_cm = np.array([cm_utt[u][0] for u in common_ids])
    e_cf = np.array([cf_utt[u][0] for u in common_ids])
    n_w  = np.array([cm_utt[u][1] for u in common_ids])
    rng = np.random.default_rng(42)
    b_cm, b_cf = [], []
    for _ in range(1000):
        idx = rng.integers(0, len(common_ids), size=len(common_ids))
        b_cm.append(100.0 * e_cm[idx].sum() / n_w[idx].sum())
        b_cf.append(100.0 * e_cf[idx].sum() / n_w[idx].sum())
    diffs = e_cf - e_cm
    nz = diffs[diffs != 0]
    p_val = wilcoxon(nz).pvalue if len(nz) > 0 else 1.0
    print("\n" + "=" * 90)
    print(f"STATISTICAL VALIDATION (N={len(common_ids)} utterances, 1000 Bootstrap resamples)")
    print(f"  ConMamba-Small 95% CI : [{np.percentile(b_cm, 2.5):.2f}%, {np.percentile(b_cm, 97.5):.2f}%]")
    print(f"  Conformer-Small 95% CI: [{np.percentile(b_cf, 2.5):.2f}%, {np.percentile(b_cf, 97.5):.2f}%]")
    print(f"  Paired Wilcoxon p-val : {p_val:.5f} | ConMamba <= Conformer on {100*(e_cm <= e_cf).mean():.1f}% utts")
    print("=" * 90)

# --- G. Export IEEE TASLP Booktabs LaTeX ---
latex_out = []
latex_out.append(r"% === TABLE: ARCHITECTURAL ABLATION ===")
latex_out.append(r"\begin{table}[t]\centering")
latex_out.append(r"\caption{Architectural ablation of ConMamba-Small on EJUST Egyptian Arabic.}")
latex_out.append(r"\begin{tabular}{lcccc}\toprule")
latex_out.append(r"\textbf{Configuration} & \textbf{Params} & \textbf{Test WER} & \textbf{Test CER} & \textbf{Long WER} \\ \midrule")
for _, r in df_abl.iterrows():
    v = str(r["Variant"]).replace("_", "\\_")
    latex_out.append(f"{v} & {r['Params (M)']}M & {r['Test WER (%)']} & {r['Test CER (%)']} & {r['Long WER (%)']} \\\\")
latex_out.append(r"\bottomrule\end{tabular}\end{table}")

if noise_rows:
    latex_out.append("\n% === TABLE: NOISE ROBUSTNESS (SNR SWEEP) ===")
    latex_out.append(r"\begin{table}[t]\centering")
    latex_out.append(r"\caption{Noise robustness WER (\%) under multi-talker babble and ambient noise.}")
    latex_out.append(r"\begin{tabular}{lccccc}\toprule")
    latex_out.append(r"\textbf{Model} & \textbf{Clean} & \textbf{20 dB} & \textbf{10 dB} & \textbf{5 dB} & \textbf{0 dB} \\ \midrule")
    for _, r in df_noise.iterrows():
        m = str(r["Model"]).replace("_", "\\_")
        latex_out.append(f"{m} & {r['Clean']} & {r['20 dB']} & {r['10 dB']} & {r['5 dB']} & {r['0 dB']} \\\\")
    latex_out.append(r"\bottomrule\end{tabular}\end{table}")

latex_text = "\n".join(latex_out)
Path(f"{RESULTS_ROOT}/q1_tables.tex").write_text(latex_text)
print("\n" + latex_text)
print(f"\nSaved all Q1 tables & logs to: {RESULTS_ROOT}")

Python 3.13.16 | PyTorch 2.11.0+cu130 | GPUs: 2x Tesla T4
Patched Mamba-ASR successfully.
Dataset root: /kaggle/input/notebooks/monaygad/egyptian-arabic-asr-masry-prepared/ejust_prepared_v1

=== Dispatching 6 Q1 Jobs across 2 GPU(s) ===
[GPU 1] SKIP conformer_small_s7775: skip_train=True but no checkpoint found in /kaggle/working/q1_new_results/conformer_small_s7775/save.[GPU 0] SKIP conmamba_small_s7775: skip_train=True but no checkpoint found in /kaggle/working/q1_new_results/conmamba_small_s7775/save.

[GPU 0] START Whisper-Small LoRA Fine-Tuning (5 epochs)...[ABLATION] conmamba_unidir_s7775.yaml: set bidirectional -> False (2 match)

[GPU 1] START conmamba_unidir_s7775 (skip_train=False) -> /kaggle/working/q1_new_results/conmamba_unidir_s7775
Traceback (most recent call last):
[WARN] Ablation key 'kernel_size' not found in conmamba_small.yaml; skipping override.
[GPU 0] START conmamba_k7_s7775 (skip_train=False) -> /kaggle/working/q1_new_results/conmamba_k7_s7775
[conmamba_unidir_s

ErrorDuringImport: problem in speechbrain - AttributeError: module 'torchaudio' has no attribute 'list_audio_backends'